# N5 - Cross-encoder rerank top candidates (GPU)

Kaggle settings: **Internet ON**, accelerator **GPU T4 x2**. Add inputs: `amz-er-2026-raw` + `amz-er-2026-code` + previous stage outputs. See `kaggle/PUSH_INSTRUCTIONS.md` in the repo.

In [ ]:
!pip -q install sentence-transformers || pip -q install --no-index --find-links /kaggle/input/amz-er-2026-wheels sentence-transformers

In [ ]:
import glob
import importlib
import os
import sys
import zipfile
from pathlib import Path

os.environ["BER_ARTIFACT_DIR"] = "/kaggle/working/artifacts"
_whl = sorted(glob.glob("/kaggle/input/**/*.whl", recursive=True))


def _locate_code():
    hits = sorted(glob.glob("/kaggle/input/**/ber/__init__.py", recursive=True))
    return str(Path(hits[0]).parent.parent) if hits else ""


def _locate_raw():
    hits = sorted(glob.glob("/kaggle/input/**/train/train_source1.tsv", recursive=True))
    return str(Path(hits[0]).parent.parent) if hits else "/kaggle/input/amz-er-2026-raw"


CODE = _locate_code()
RAW = _locate_raw()
os.environ["BER_DATA_DIR"] = RAW


def _vendor():
    dst = Path("/kaggle/working/_deps")
    dst.mkdir(parents=True, exist_ok=True)
    for w in _whl:
        try:
            zipfile.ZipFile(w).extractall(dst)
        except Exception as exc:
            print("wheel extract failed", w, exc)
    return str(dst)


DEPS = _vendor()
for _p in (DEPS, CODE):
    if _p:
        sys.path.insert(0, _p)
print("code:", CODE)
print("raw:", RAW)
print("wheels:", len(_whl))
try:
    import anyascii, jellyfish, rapidfuzz
    print("deps OK")
except Exception as exc:
    print("deps FAILED:", exc)
try:
    import ber
    print("ber OK")
except Exception as exc:
    print("ber FAILED:", exc)


def find(sub):
    hits = sorted(glob.glob(f"/kaggle/input/**/artifacts/{sub}", recursive=True))
    print(sub, "->", hits[:2])
    return hits[0] if hits else ""


CLEAN = find("clean")
BLOCK = find("block")
EMBED = find("embed")
GBDT = find("gbdt")
RERANK = find("rerank")


def run(module, *args):
    mod = importlib.import_module(module)
    argv = sys.argv
    sys.argv = [module] + [str(a) for a in args]
    try:
        mod.main()
    finally:
        sys.argv = argv

In [ ]:
import socket
try:
    socket.create_connection(("pypi.org", 443), timeout=10)
    print("internet: OK")
except Exception as exc:
    print("internet: FAILED", exc)

In [ ]:
run("ber.stages.rerank", "--clean-dir", CLEAN, "--gbdt-dir", GBDT, "--out-dir", ART + "/rerank")

In [ ]:
import json
import os
from pathlib import Path

ART = os.environ["BER_ARTIFACT_DIR"]
for sub in ("clean", "block", "embed", "gbdt", "rerank", "out"):
    for name in ("metrics.json", "stats.json"):
        p = Path(ART) / sub / name
        if p.exists():
            print("==", sub, name, "==")
            print(p.read_text(encoding="utf-8"))

out_dir = Path(ART) / "out"
if out_dir.exists():
    print("outputs:", sorted(x.name for x in out_dir.glob("*.tsv")))